## 1. `bam_to_cz` vs `bam_to_allc`

In [7]:
import os,sys
import pandas as pd
os.chdir(os.path.expanduser("~/Projects/test_cytozip"))

In [2]:
from ALLCools._bam_to_allc import bam_to_allc
%time bam_to_allc(bam_path="cytozip_example_data/hg38_bam/UWA7648_CX1819_NAC_1_P10-1-K18-A10.bam", \
                  reference_fasta=os.path.expanduser("~/Ref/hg38/hg38_ucsc_with_chrL.fa"), \
                  output_path="cytozip_example_data/hg38_bam/UWA7648_CX1819_NAC_1_P10-1-K18-A10.allc.tsv.gz", \
                  chroms=os.path.expanduser("~/Ref/hg38/hg38_ucsc_with_chrL.main.chrom.sizes"))

[W::hts_idx_load3] The index file is older than the data file: cytozip_example_data/hg38_bam/UWA7648_CX1819_NAC_1_P10-1-K18-A10.bam.bai
[W::hts_idx_load3] The index file is older than the data file: cytozip_example_data/hg38_bam/UWA7648_CX1819_NAC_1_P10-1-K18-A10.bam.bai


CPU times: user 2min 40s, sys: 13.5 s, total: 2min 54s
Wall time: 3min 9s


,mc,cov,mc_rate,genome_cov
CTT,42937,2689050,0.015967,0.040206
CCT,25755,2274939,0.011321,0.040206
CAC,307965,2013638,0.152940,0.040206
CAG,159326,2715612,0.058670,0.040206
CCA,27004,2462628,0.010966,0.040206
CTG,40374,2644319,0.015268,0.040206
CCC,20390,1641759,0.012420,0.040206
CGC,244843,293296,0.834798,0.040206
CCG,5368,354632,0.015137,0.040206
CAA,111157,2702079,0.041138,0.040206


In [6]:
from cytozip.bam import bam_to_cz
%time bam_to_cz(bam_path="cytozip_example_data/hg38_bam/UWA7648_CX1819_NAC_1_P10-1-K18-A10.bam", \
                  genome=os.path.expanduser("~/Ref/hg38/hg38_ucsc_with_chrL.fa"), \
                  output="cytozip_example_data/hg38_bam/UWA7648_CX1819_NAC_1_P10-1-K18-A10.cz", \
                  reference="~/Ref/hg38/hg38_with_chrL.allc.cz",\
                   chroms=os.path.expanduser("~/Ref/hg38/hg38_ucsc_with_chrL.main.chrom.sizes") \
               ) 
# CPU times: user 1min 21s, sys: 7.41 s, total: 1min 29s
# Wall time: 1min 31s

[W::hts_idx_load3] The index file is older than the data file: cytozip_example_data/hg38_bam/UWA7648_CX1819_NAC_1_P10-1-K18-A10.bam.bai


CPU times: user 1min 19s, sys: 7.18 s, total: 1min 27s
Wall time: 1min 29s


,mc,cov,mc_rate,genome_cov
CTT,42937,2689050,0.015967,0.008107
CCT,25755,2274939,0.011321,0.008107
CAC,307965,2013638,0.152940,0.008107
CAG,159326,2715612,0.058670,0.008107
CCA,27004,2462628,0.010966,0.008107
CTG,40374,2644319,0.015268,0.008107
CCC,20390,1641759,0.012420,0.008107
CGC,244843,293296,0.834798,0.008107
CCG,5368,354632,0.015137,0.008107
CAA,111157,2702079,0.041138,0.008107


In [8]:
# validate whether cz and allc store the same values
df_allc=pd.read_csv("cytozip_example_data/hg38_bam/UWA7648_CX1819_NAC_1_P10-1-K18-A10.allc.tsv.gz", 
                    sep="\t", header=None, usecols=[0, 1, 4, 5],names=["chrom", "pos", "mc_allc", "cov_allc"],
    )
df_allc.head()

,chrom,pos,mc_allc,cov_allc
0,chr1,14932,0,1
1,chr1,14933,0,1
2,chr1,14935,1,1
3,chr1,14938,1,1
4,chr1,14939,0,1


In [ ]:
from cytozip import Reader
from cytozip.bam import _LazyRefPositions
cell = Reader("cytozip_example_data/hg38_bam/UWA7648_CX1819_NAC_1_P10-1-K18-A10.cz")
refpos=_LazyRefPositions("~/Ref/hg38/hg38_with_chrL.allc.cz")
frames, last_pos = [], {}
for dim in cell.chunk_key2offset.keys():
    chrom = dim[0]
    arr = cell.chunk2numpy(dim)          # f0=mc, f1=cov (uint8)
    mc, cov = arr["f0"], arr["f1"]
    pos = refpos.get(chrom)              # uint32, row-aligned with mc/cov
    if pos is None:
        continue
    if len(pos) != len(mc):
        raise ValueError(
            f"{cz_path} chrom {chrom}: {len(mc)} records but reference "
            f"has {len(pos)} positions (not row-aligned).")
    last_pos[chrom] = int(pos[-1]) if len(pos) else -1
    keep = cov > 0
    frames.append(pd.DataFrame({
        "chrom": chrom,
        "pos": pos[keep].astype(np.int64),
        "mc_cz": mc[keep].astype(np.int64),
        "cov_cz": cov[keep].astype(np.int64),
    }))
    refpos.drop(chrom)                   # release this chrom's ref pages
cell.close()
cz = (pd.concat(frames, ignore_index=True) if frames
      else pd.DataFrame(columns=["chrom", "pos", "mc_cz", "cov_cz"]))

<a href="benchmark/benchmark_methylation_calling.ipynb">
    <img src="benchmark/bam_benchmark.png" title="kwargs, gap and pad" align="center" width="250px">
</a>